In [1]:
import os, sys
from tqdm import tqdm
import torch
import numpy as np
from scipy import stats
from procrustes import rotational
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.join(os.path.abspath(''), '..'))
from cmm.ffxml import ForceFieldXML
from cmm.topology import Topology
from cmm.units import BOHR2NM, BOHR2ANG, HARTREE2KCAL, AMU2ELECTRON_MASS, HARTREE2WAVENUMBER
from cmm.drivers import OptimizationDriver, HarmonicAnalysisDriver
from cmm.misc_utils import read_xyz, write_xyz, get_masses

import openmm.app as app
from ase.io import read

In [2]:
def rmsd(A: np.ndarray, B: np.ndarray):
    return np.sqrt(np.mean(np.sum((A - B)**2, axis=1)))

In [3]:
torch.set_default_dtype(torch.float64)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
ff_path = os.path.join(os.path.abspath(''), '../scripts/ion_water_params_8_19.xml')
ff = ForceFieldXML(ff_path, device=device)

In [4]:
water_cluster_pdb_path = os.path.join(os.path.abspath(''), '../tests/data/water_clusters/all_reference_clusters.pdb')
water_cluster_xyz_path = os.path.join(os.path.abspath(''), '../tests/data/water_clusters/all_reference_clusters.xyz')

f_water_scan_pdb_path = os.path.join('/home/heindelj/research/Teresa/reactive_force_fields/reference_data/CMM_Data/ion_water/ion_water_scans/h2o_f_scan.pdb')
cl_water_scan_pdb_path = os.path.join('/home/heindelj/research/Teresa/reactive_force_fields/reference_data/CMM_Data/ion_water/ion_water_scans/h2o_cl_scan.pdb')
br_water_scan_pdb_path = os.path.join('/home/heindelj/research/Teresa/reactive_force_fields/reference_data/CMM_Data/ion_water/ion_water_scans/h2o_br_scan.pdb')
i_water_scan_pdb_path = os.path.join('/home/heindelj/research/Teresa/reactive_force_fields/reference_data/CMM_Data/ion_water/ion_water_scans/h2o_i_scan.pdb')

li_water_scan_pdb_path = os.path.join('/home/heindelj/research/Teresa/reactive_force_fields/reference_data/CMM_Data/ion_water/ion_water_scans/h2o_li_scan.pdb')
na_water_scan_pdb_path = os.path.join('/home/heindelj/research/Teresa/reactive_force_fields/reference_data/CMM_Data/ion_water/ion_water_scans/h2o_na_scan.pdb')
k_water_scan_pdb_path = os.path.join('/home/heindelj/research/Teresa/reactive_force_fields/reference_data/CMM_Data/ion_water/ion_water_scans/h2o_k_scan.pdb')
rb_water_scan_pdb_path = os.path.join('/home/heindelj/research/Teresa/reactive_force_fields/reference_data/CMM_Data/ion_water/ion_water_scans/h2o_rb_scan.pdb')
cs_water_scan_pdb_path = os.path.join('/home/heindelj/research/Teresa/reactive_force_fields/reference_data/CMM_Data/ion_water/ion_water_scans/h2o_cs_scan.pdb')

mg_water_scan_pdb_path = os.path.join('/home/heindelj/research/Teresa/reactive_force_fields/reference_data/CMM_Data/ion_water/ion_water_scans/h2o_mg_scan.pdb')
ca_water_scan_pdb_path = os.path.join('/home/heindelj/research/Teresa/reactive_force_fields/reference_data/CMM_Data/ion_water/ion_water_scans/h2o_ca_scan.pdb')

In [5]:
water_cluster_pdb = app.PDBFile(water_cluster_pdb_path)
positions = [water_cluster_pdb.getPositions(True, frame=i)._value * 10.0 for i in range(water_cluster_pdb.getNumFrames())]

topologies = [Topology.fromMultiPDB(water_cluster_pdb_path, device, frame_index=i) for i in range(water_cluster_pdb.getNumFrames())]
systems = [ff.parametrize(topology, use_fd_morse=True, periodic=False, use_lr_dispersion=False, use_cutoff=False) for topology in topologies]

topologies_no_fd = [Topology.fromMultiPDB(water_cluster_pdb_path, device, frame_index=i) for i in range(water_cluster_pdb.getNumFrames())]
systems_no_fd = [ff.parametrize(topology, use_fd_morse=False, periodic=False, use_lr_dispersion=False, use_cutoff=False) for topology in topologies_no_fd]

topologies_with_fd = [Topology.fromMultiPDB(water_cluster_pdb_path, device, frame_index=i) for i in range(water_cluster_pdb.getNumFrames())]
systems_with_fd = [ff.parametrize(topology, use_fd_morse=True, periodic=False, use_lr_dispersion=False, use_cutoff=False) for topology in topologies_with_fd]

reference_coords, reference_labels = read_xyz(water_cluster_xyz_path)
reference_masses = [get_masses(reference_labels[i]) for i in range(len(reference_labels))]

/home/heindelj/miniforge3/envs/pycmm/lib/python3.12/site-packages/torch/nested/__init__.py:226: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at ../aten/src/ATen/NestedTensorImpl.cpp:178.)
  return _nested.nested_tensor(


In [6]:
def create_systems(pdb_path: str):
    pdb_openmm = app.PDBFile(pdb_path)
    positions = [pdb_openmm.getPositions(True, frame=i)._value * 10.0 for i in range(pdb_openmm.getNumFrames())]
    topologies = [Topology.fromMultiPDB(pdb_path, device, frame_index=i) for i in range(pdb_openmm.getNumFrames())]
    systems = [ff.parametrize(topology, use_fd_morse=True, periodic=False, use_lr_dispersion=False, use_cutoff=False) for topology in topologies]
    return positions, systems

In [7]:
f_water_positions, f_water_scan_systems = create_systems(f_water_scan_pdb_path)
cl_water_positions, cl_water_scan_systems = create_systems(cl_water_scan_pdb_path)
br_water_positions, br_water_scan_systems = create_systems(br_water_scan_pdb_path)
i_water_positions, i_water_scan_systems = create_systems(i_water_scan_pdb_path)

li_water_positions, li_water_scan_systems = create_systems(li_water_scan_pdb_path)
na_water_positions, na_water_scan_systems = create_systems(na_water_scan_pdb_path)
k_water_positions, k_water_scan_systems = create_systems(k_water_scan_pdb_path)
rb_water_positions, rb_water_scan_systems = create_systems(rb_water_scan_pdb_path)
cs_water_positions, cs_water_scan_systems = create_systems(cs_water_scan_pdb_path)

mg_water_positions, mg_water_scan_systems = create_systems(mg_water_scan_pdb_path)
ca_water_positions, ca_water_scan_systems = create_systems(ca_water_scan_pdb_path)

In [8]:
ions_systems_and_positions = [
    ("li+", li_water_positions[7], li_water_scan_systems[7]),
    ("na+", na_water_positions[7], na_water_scan_systems[7]),
    ("k+", k_water_positions[7], k_water_scan_systems[7]),
    ("rb+", rb_water_positions[7], rb_water_scan_systems[7]),
    ("cs+", cs_water_positions[7], cs_water_scan_systems[7]),
    ("mg2+", mg_water_positions[7], mg_water_scan_systems[7]),
    ("ca2+", ca_water_positions[7], ca_water_scan_systems[7]),
    ("f-", f_water_positions[7], f_water_scan_systems[7]),
    ("cl-", cl_water_positions[7], cl_water_scan_systems[7]),
    ("br-", br_water_positions[7], br_water_scan_systems[7]),
    ("i-", i_water_positions[7], i_water_scan_systems[7])
]

def optimize_ion_water_dimers_and_compute_frequencies(data_in):
    optimized_coords = []
    optimized_energies = []
    harmonic_frequencies = []
    for i in range(len(data_in)):
        label, positions, system = data_in[i]
        opt_driver = OptimizationDriver(system)
        coords = torch.from_numpy(positions / BOHR2ANG).to(device).requires_grad_(False)
        box = torch.tensor(np.eye(3) * 100.0, requires_grad=False, device=device)
        coords_opt, box_opt, result = opt_driver.run(coords, box)
        energy = result.fun * HARTREE2KCAL
        print(f"Optimized H2O...{label}: E = {energy}")
        optimized_coords.append(coords_opt)
        optimized_energies.append(energy)
        
        harmonic_driver = HarmonicAnalysisDriver(system)
        masses = get_masses(system.top._atom_symbols)
        hessian = harmonic_driver.run(coords_opt.to(device).requires_grad_(False), box, masses * AMU2ELECTRON_MASS)
        eigvals, eigvecs = torch.linalg.eigh(hessian)
        freqs = torch.sqrt(eigvals) * HARTREE2WAVENUMBER
        nan_mask = torch.isnan(freqs)
        nan_indices = torch.nonzero(nan_mask)
        freqs[nan_indices] = torch.zeros(nan_indices.size())
        harmonic_frequencies.append(freqs)

    return optimized_coords, optimized_energies, harmonic_frequencies

In [9]:
optimized_ion_water_coords, optimized_ion_water_energies, ion_water_dimer_freqs = optimize_ion_water_dimers_and_compute_frequencies(ions_systems_and_positions)

Optimized H2O...li+: E = -34.762954232088525
Optimized H2O...na+: E = -24.2124759203865
Optimized H2O...k+: E = -17.554389471139448
Optimized H2O...rb+: E = -15.387097791791225
Optimized H2O...cs+: E = -13.858960322976293
Optimized H2O...mg2+: E = -91.54104403748214
Optimized H2O...ca2+: E = -60.67109665532125
Optimized H2O...f-: E = -29.54422212004515
Optimized H2O...cl-: E = -15.34241943732602
Optimized H2O...br-: E = -13.404743569462221
Optimized H2O...i-: E = -11.208405057945404


In [11]:
ion_water_dimer_freqs

[tensor([       nan,        nan,        nan, 2.4461e-05, 7.7333e+01, 1.4246e+02,
         1.8547e+02, 6.3716e+02, 6.7352e+02, 1.7415e+03, 3.7691e+03, 3.8575e+03]),
 tensor([       nan,        nan, 1.8317e-05, 3.6910e+01, 6.6137e+01, 6.7787e+01,
         2.6183e+02, 3.4718e+02, 5.2701e+02, 1.7196e+03, 3.7825e+03, 3.8755e+03]),
 tensor([       nan,        nan, 8.5741e-06, 2.4662e+01, 4.0752e+01, 5.5465e+01,
         2.3463e+02, 2.8899e+02, 4.5143e+02, 1.7044e+03, 3.7910e+03, 3.8873e+03]),
 tensor([       nan,        nan, 4.0239e-06, 1.9439e+01, 3.1799e+01, 5.2506e+01,
         1.9192e+02, 2.9234e+02, 4.1874e+02, 1.6989e+03, 3.7945e+03, 3.8921e+03]),
 tensor([       nan,        nan, 2.4760e-05, 1.7256e+01, 2.7879e+01, 4.6748e+01,
         1.6672e+02, 2.8907e+02, 4.0547e+02, 1.6954e+03, 3.7972e+03, 3.8955e+03]),
 tensor([       nan,        nan, 7.9000e-06, 2.7267e-05, 1.6336e+02, 2.2442e+02,
         2.8864e+02, 8.1706e+02, 1.0080e+03, 1.9093e+03, 3.6152e+03, 3.6723e+03]),
 tensor([       

In [10]:
harmonic_driver = HarmonicAnalysisDriver(systems[geom_index])
hessian = harmonic_driver.run(torch.from_numpy(coords_opt[geom_index] / BOHR2ANG).to(device).requires_grad_(False), box, masses[geom_index] * AMU2ELECTRON_MASS)
eigvals, eigvecs = torch.linalg.eigh(hessian)
freqs = torch.sqrt(eigvals) * HARTREE2WAVENUMBER

NameError: name 'geom_index' is not defined

In [ ]:
print(optimized_ion_water_coords[0])

tensor([[ 1.3184, -1.7068, -1.1438],
        [ 1.7776, -0.4343,  0.0839],
        [-0.4920, -1.8381, -0.9371],
        [ 3.3883, -3.4570, -3.3377]])


In [ ]:
rmsd_i = rmsd(result.new * BOHR2ANG, i_water_positions[7])

AttributeError: new

In [ ]:
coords = torch.from_numpy(positions[1] / BOHR2ANG).to(device).requires_grad_(False)
box = torch.tensor(np.eye(3) * 100.0, requires_grad=False, device=device)
energies = systems[1].getEnergy(coords, box)

for key in energies:
    print(f"{key}: {energies[key] * HARTREE2KCAL} kcal/mol")

bond: 0.40315064636088754 kcal/mol
angle: 0.023446173018395126 kcal/mol
torsion: 0.0 kcal/mol
bond_bond: -0.005611845671081472 kcal/mol
bond_angle: 0.03452899245356382 kcal/mol
angle_angle: 0.0 kcal/mol
torsion_bond: 0.0 kcal/mol
torsion_angle: 0.0 kcal/mol
torsion_angle_angle: 0.0 kcal/mol
perm_elec: -25.909409713076034 kcal/mol
pol: -3.7103559196676774 kcal/mol
ct_direct: -6.5340623624889105 kcal/mol
xpol: -0.6042721511840821 kcal/mol
pauli: 27.69132165343953 kcal/mol
disp: -6.1386630971369645 kcal/mol
total: -14.749927623952374 kcal/mol


In [ ]:
opt_coords_path_with_fd = os.path.join(os.path.abspath(''), 'reference_opt_cmm_with_fd_morse.xyz')
coords_opt, labels = read_xyz(opt_coords_path_with_fd)

all_perm_elec = []
all_pauli = []
all_dispersion = []
all_induction = []
all_total = []

for i in range(len(coords_opt)):
    coords = torch.from_numpy(coords_opt[i] / BOHR2ANG).to(device).requires_grad_(False)
    box = torch.tensor(np.eye(3) * 100.0, requires_grad=False, device=device)
    energies = systems[i].getEnergy(coords, box)

    all_perm_elec.append(energies['perm_elec'].item() * HARTREE2KCAL)
    all_pauli.append(energies['pauli'].item() * HARTREE2KCAL)
    all_dispersion.append(energies['disp'].item() * HARTREE2KCAL)
    all_induction.append((energies['pol'] + energies['ct_direct']).item() * HARTREE2KCAL)
    all_total.append(energies['total'].item() * HARTREE2KCAL)
print(all_perm_elec)
print(all_pauli)
print(all_dispersion)
print(all_induction)
print(all_total)

[-8.607751904867047, -29.296102363079417, -53.133407123209636, -69.63209919679979, -83.76533898419414, -84.6497181683305, -87.19425422091291, -85.48081909454606, -106.97921260863626, -136.7261123046401, -137.73332732188746, -154.33353363528556, -176.6097887451441, -192.82459788749833, -306.15923709163206, -298.8574233714669, -298.57354001010566, -309.2916417198375, -309.22847204294925, -321.30726290103627, -395.13796319763526, -382.3757679444182, -380.2235906331387, -380.5292211513273, -508.20800984208773]
[8.60292772403882, 33.80011569437262, 65.51194823851014, 87.8569141870256, 101.11901807810978, 103.74855552079633, 109.35969274784436, 107.85599527020923, 132.64066964873672, 169.3759556896411, 172.45607436024247, 193.54383452155832, 222.87651453663847, 244.15243153740542, 390.23175954580324, 375.4633960639609, 376.45180566495713, 394.20458526774934, 396.3080187028132, 411.1973604935507, 509.1241220376364, 486.4984318331419, 478.35868425206877, 487.98539901585127, 660.0747386860592]


In [ ]:
for energy in all_total:
    print(energy)

-4.857254338407799
-15.14928952119107
-27.351428332984636
-36.12704589743227
-45.2513595195833
-45.07149296906773
-45.487250591740874
-44.72013802136345
-57.08232843007485
-72.37098594130718
-72.54078489262719
-82.0266289372716
-93.61141029070305
-102.85163572166937
-164.3453375119729
-162.84962741831856
-162.64251188186884
-164.14560662047782
-164.42058780518036
-174.96524428878215
-212.84587144957877
-208.90096764731823
-208.156745608938
-200.74584718580664
-273.1349713467862


In [ ]:
def optimize_all_reference_structures_and_compute_rmsds(systems, reference_coords):
    optimized_coords = []
    optimized_energies = []
    rmsds = []
    for i in range(len(systems)):
        opt_driver = OptimizationDriver(systems[i])
        coords = torch.from_numpy(reference_coords[i] / BOHR2ANG).to(device).requires_grad_(False)
        box = torch.tensor(np.eye(3) * 100.0, requires_grad=False, device=device)
        coords_opt, box_opt, result = opt_driver.run(coords, box)
        optimized_energies.append(result.fun * HARTREE2KCAL)
        if result.success == False:
            print(f"Warning: Optimization of structure {i} did not converge. Final energy was {optimized_energies[-1]} kcal/mol.")
        # Get RMSD and store new coordinates.
        # Align structures first in case they rotated during optimization.
        coords_opt = coords_opt.numpy() * BOHR2ANG
        result = rotational(coords_opt, reference_coords[i])
        optimized_coords.append(result.new_a)
        rmsd_i = rmsd(result.new_a, reference_coords[i])
        rmsds.append(rmsd_i)
        print(f"Structure {i}: Energy = {optimized_energies[-1]} kcal/mol, RMSD = {rmsd_i} Ang.")
    
    return optimized_coords, optimized_energies, rmsds

In [ ]:
#opt_coords, opt_energies, opt_rmsds = optimize_all_reference_structures_and_compute_rmsds(systems, reference_coords)

# Write out the optimized coords to a file
#write_xyz("reference_opt_cmm_with_fd_morse.xyz", reference_labels, opt_coords)

In [ ]:
def compute_hbond_distance_freq_correlation_for_ref_clusters(systems, optimized_coord_file):
    all_dists = []
    all_freqs = []

    coords_opt, labels = read_xyz(optimized_coord_file)
    masses = [get_masses(labels[i]) for i in range(len(labels))]
    box = torch.tensor(np.eye(3) * 100.0, requires_grad=False, device=device)
    for geom_index in tqdm(range(len(coords_opt))):
        box = torch.tensor(np.eye(3) * 100.0, requires_grad=False, device=device)

        harmonic_driver = HarmonicAnalysisDriver(systems[geom_index])
        hessian = harmonic_driver.run(torch.from_numpy(coords_opt[geom_index] / BOHR2ANG).to(device).requires_grad_(False), box, masses[geom_index] * AMU2ELECTRON_MASS)
        eigvals, eigvecs = torch.linalg.eigh(hessian)
        freqs = torch.sqrt(eigvals) * HARTREE2WAVENUMBER

        # Find the atom which moves most for each hbond frequency
        minimum_hbond_frequency = 2500.0
        maximum_hbond_frequency = 3800.0
        mask = (freqs >= minimum_hbond_frequency) & (freqs <= maximum_hbond_frequency)
        hbond_indices = torch.nonzero(mask).squeeze()
        hbond_eigvces = eigvecs[:, hbond_indices]
        if hbond_eigvces.dim() == 1:
            hbond_eigvces.unsqueeze_(1)
        hbond_freqs = freqs[hbond_indices]
        if hbond_freqs.ndim == 0:
            hbond_freqs = hbond_freqs[np.newaxis]

        h_indices = torch.zeros(hbond_eigvces.size(-1), dtype=torch.long)
        for i in range(hbond_eigvces.size(-1)):
            mode = hbond_eigvces[:, i].reshape(-1, 3)
            h_index = torch.argmax(torch.linalg.norm(mode, dim=1))
            h_indices[i] = h_index

        # Get the distances between each hydrogen and the other oxygens in the system
        all_O_indices = []
        all_H_indices = []
        for i_geom in range(len(labels)):
            O_indices = []
            H_indices = []
            for i, label in enumerate(labels[i_geom]):
                if label == 'O':
                    O_indices.append(i)
                if label == 'H':
                    H_indices.append(i)
            all_O_indices.append(O_indices)
            all_H_indices.append(H_indices)

        all_O_indices = [torch.tensor(all_O_indices[i]) for i in range(len(all_O_indices))]
        all_H_indices = [torch.tensor(all_H_indices[i]) for i in range(len(all_H_indices))]

        for index in h_indices:
            if reference_labels[geom_index][index] != 'H':
                print("Warning: Found an h-bond frequency where the most mobile atom was not hydrogen.")

        H_atoms_hbond_only = coords_opt[geom_index][h_indices]
        O_atoms = coords_opt[geom_index][all_O_indices[geom_index]]
        if H_atoms_hbond_only.ndim == 1:
            H_atoms_hbond_only = H_atoms_hbond_only[np.newaxis, :]
            
        dist_vecs = H_atoms_hbond_only[:, np.newaxis, :] - O_atoms[np.newaxis, :, :]
        OH_hbond_dists = np.array([np.min(np.linalg.norm(dist_vecs[i], axis=1)) for i in range(dist_vecs.shape[0])])
        #OH_hbond_dists_sorted = np.sort(OH_hbond_dists)[::-1]
        all_dists.append(OH_hbond_dists)#_sorted)
        all_freqs.append(hbond_freqs.numpy())
    return np.concatenate(all_dists), np.concatenate(all_freqs)

In [ ]:
#opt_coords_path_no_fd = os.path.join(os.path.abspath(''), 'reference_opt_cmm_without_fd_morse.xyz')
#all_dists_no_fd, all_freqs_no_fd = compute_hbond_distance_freq_correlation_for_ref_clusters(systems_no_fd, opt_coords_path_no_fd)
#
#opt_coords_path_with_fd = os.path.join(os.path.abspath(''), 'reference_opt_cmm_with_fd_morse.xyz')
#all_dists_with_fd, all_freqs_with_fd = compute_hbond_distance_freq_correlation_for_ref_clusters(systems_with_fd, opt_coords_path_with_fd)

In [ ]:
def plot_badger_rule_correlation(r_hbond_no_fd, freqs_hbond_no_fd,
                                 r_hbond_with_fd, freqs_hbond_with_fd,
                                 title="Badger Rule Correlation"):
    r_eq_cmm = 0.9589289
    freq_ave_cmm = (3945.054377 + 3834.691479) / 2

    r_eq_wb97xv = 0.959274
    freq_ave_wb97xv = (3960.83 + 3859.85) / 2

    shifted_r_cmm_no_fd = r_hbond_no_fd - r_eq_cmm
    shifted_freqs_cmm_no_fd = freqs_hbond_no_fd - freq_ave_cmm

    shifted_r_cmm_with_fd = r_hbond_with_fd - r_eq_cmm
    shifted_freqs_cmm_with_fd = freqs_hbond_with_fd - freq_ave_cmm

    slope_no_fd, intercept_no_fd, _, _, _ = stats.linregress(shifted_r_cmm_no_fd, shifted_freqs_cmm_no_fd)
    slope_with_fd, intercept_with_fd, _, _, _ = stats.linregress(shifted_r_cmm_with_fd, shifted_freqs_cmm_with_fd)
    
    line_x_no_fd = np.linspace(shifted_r_cmm_no_fd.min(), shifted_r_cmm_no_fd.max(), 100)
    line_y_no_fd = slope_no_fd * line_x_no_fd + intercept_no_fd

    line_x_with_fd = np.linspace(shifted_r_cmm_with_fd.min(), shifted_r_cmm_with_fd.max(), 100)
    line_y_with_fd = slope_with_fd * line_x_with_fd + intercept_with_fd
    
    plt.figure(figsize=(10, 6))
    plt.scatter(shifted_r_cmm_no_fd, shifted_freqs_cmm_no_fd, alpha=0.7, color='blue', s=50, label="No FD Morse")
    plt.plot(line_x_no_fd, line_y_no_fd, color='blue', linewidth=2, linestyle="--",
             label=f'Linear fit: y = {slope_no_fd:.2f}x + {intercept_no_fd:.2f}')
    
    plt.scatter(shifted_r_cmm_with_fd, shifted_freqs_cmm_with_fd, alpha=0.7, color='green', s=50, label="With FD Morse")
    plt.plot(line_x_with_fd, line_y_with_fd, color='green', linewidth=2, linestyle="--",
         label=f'Linear fit: y = {slope_with_fd:.2f}x + {intercept_with_fd:.2f}')
    
    plt.xlabel('OH Bond Shift (Å)')
    plt.ylabel('Frequency Shift (cm^-1)')
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print("Linear Regression Results:")
    print(f"Slope No FD: {slope_no_fd:.4f}")
    print(f"Intercept No FD: {intercept_no_fd:.4f}")
    #print(f"R-squared: {r_value**2:.4f}")
    #print(f"Correlation coefficient: {r_value:.4f}")
    #print(f"P-value: {p_value:.4e}")
    #print(f"Standard error: {std_err:.4f}")

In [ ]:
#plot_badger_rule_correlation(all_dists_no_fd, all_freqs_no_fd, all_dists_with_fd, all_freqs_with_fd)